# OCRia-1 4B — document parsing on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/OCRia-1-document-parsing.ipynb)

[`llamaindex/OCRia-1`](https://huggingface.co/llamaindex/OCRia-1) is LlamaIndex's **4B** document-parsing vision-language model, fine-tuned from [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B). It reads **one page image** and writes the page as **Markdown**, with tables and chart data as **HTML**. No layout detector, OCR engine, or PDF text layer. The Hub file is the unoptimized **BF16** weights (`model.safetensors`, **10.35 GB**), public and not gated.

**Why this model.** It is built for the page types document parsers still get wrong: merged cells and spanning headers, long tables, charts whose numbers have to be read off the plot, and formatting that carries meaning (strikethrough, underline, checkboxes, sub/superscript). The card: highest-scoring open-weight model on both [ParseBench](https://parsebench.ai) leaderboards with visual grounding included — **85.24** on ParseBench 2 (7 of 61 systems; **1 of 23** open-weight) and **84.65** on ParseBench 1 (2 of 158; **1 of 53** open-weight). ParseBench 1 charts: **82.47**. Those are the card's numbers, not this notebook.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/OCRia-1-document-parsing.md) for when this notebook is a good fit and when it is not.

**Licence:** `other` — [OCRia-1 Model and Code License](https://huggingface.co/llamaindex/OCRia-1/blob/main/LICENSE). Free for research (academic/non-profit research, teaching, evaluation, and internal R&D that does not reach a product). Free for personal use, and for **commercial use by organizations under $2M annual revenue and under $2M total funding**. A commercial license is required above those thresholds, and for any document-parsing / OCR / extraction product that competes with LlamaParse or LlamaCloud, whatever your size. You own the outputs. Base model Qwen3.5-4B is Apache-2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about **15 GB** of VRAM, **no native bfloat16**, and **no FlashAttention 2**. The card's Transformers snippet loads `dtype=torch.bfloat16`; this notebook loads **float16**. If float16 does not fit, it falls back to **8-bit** then **4-bit** bitsandbytes. Hub weights are **10.35 GB**. Pages are downscaled to about **150 DPI / max side 1600 px** (the card's example pages were rendered at 150 DPI; its quality recipe is 300 DPI / ~4.1 megapixels). `max_new_tokens` is **4096**, not the card's 24,576. Peak VRAM is printed at the end; it has **not** been measured on a real T4 in this repo.

**Not in this notebook:** vLLM, LlamaParse, layout boxes (`ocria_grounding`), Ollama, or the [GGUF builds](https://huggingface.co/llamaindex/OCRia-1-GGUF) (llama.cpp / Ollama — an alternative if you want local CPU/GPU inference). Do not install `flash-attn`. Do not reinstall Colab's torch.

**Links:** [model card](https://huggingface.co/llamaindex/OCRia-1) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/OCRia-1-document-parsing.md) · [report](https://arxiv.org/abs/2611.00092) · [announcement](https://x.com/llama_index/status/2125315040723062872) · [GGUF](https://huggingface.co/llamaindex/OCRia-1-GGUF)


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate>=1.2.0" "bitsandbytes>=0.45.0" "huggingface_hub>=0.23.0" "pypdfium2==5.14.0" pillow

# Card: pip install transformers accelerate torchvision (Transformers 5.3 or later).
# config.json records transformers_version 5.3.0. This notebook pins 5.20.1, the
# same Qwen3.5 VLM pin as LightOnOCR-3 0.8B / 4B in this repo (WePocketModel
# pins 5.3.0 to match that project's GitHub; OmniEVIE uses transformers>=5.13.1).
# bitsandbytes: 8-bit / 4-bit fallback if float16 does not fit ~15 GB.
# pypdfium2 renders the table-page PDF. pillow is already on Colab.
# Use %pip, not `python -I -m pip`: isolated mode drops Colab site-packages.
# Do NOT reinstall torch. Do NOT install flash-attn.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to 5.20.1 is invisible until the process restarts.


In [ ]:
import os

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch
import transformers
from packaging.version import Version

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

assert Version(transformers.__version__.split("+")[0]) >= Version("5.3.0"), (
    f"transformers {transformers.__version__} is loaded; OCRia-1 needs 5.3+. "
    "Run the install cell, then Runtime > Restart session."
)

# T4 (sm_75) has no FlashAttention 2 and no native bfloat16.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Card snippet uses torch.bfloat16. T4 is Turing sm_75 — stay on float16.
DTYPE = torch.float16
free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager)")
if major < 8:
    print("T4 path: float16 first, then 8-bit / 4-bit if load OOMs.")


def print_vram(label=""):
    suffix = f" ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM{suffix}: allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | "
        f"peak {peak:.2f} GB | free {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


## 1. Load OCRia-1

The card's **With Hugging Face Transformers** path: `AutoModelForImageTextToText` + `AutoProcessor`, `prompt.txt` as the **system** message, the page image as the **only** user content, thinking **off**, greedy decoding (`do_sample=False`). The Qwen3.5 chat template turns thinking on by default; with it on, the card says the model answers in the wrong format.

First run downloads **10.35 GB**. T4 load is **float16 + SDPA** (eager if SDPA refuses Qwen3.5 linear-attention layers). If that OOMs, the cell retries **8-bit** then **4-bit** bitsandbytes.


In [ ]:
import gc

from huggingface_hub import hf_hub_download
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "llamaindex/OCRia-1"
MAX_NEW_TOKENS = 4096  # card uses 24576; too long for a free T4
MAX_SIDE = 1600  # ~150 DPI letter page; card examples were rendered at 150 DPI
MAX_PIXELS = 4_100_000  # card's image budget (~210 DPI letter); extra cap only

SYSTEM_PROMPT = open(hf_hub_download(MODEL_ID, "prompt.txt")).read()
print("system prompt:", len(SYSTEM_PROMPT), "chars from prompt.txt")


def is_oom(exc):
    text = f"{type(exc).__name__}: {exc}".lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in text


def load_ocr(attn, quant=None):
    kwargs = {"attn_implementation": attn, "device_map": "auto"}
    if quant == "8bit":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif quant == "4bit":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=DTYPE,
        )
    else:
        kwargs["dtype"] = DTYPE
    return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs).eval()


def try_load(quant):
    try:
        loaded = load_ocr("sdpa", quant)
        return loaded, "sdpa"
    except Exception as exc:
        if is_oom(exc):
            gc.collect()
            torch.cuda.empty_cache()
            raise
        print("sdpa load failed, retrying eager:", exc)
        gc.collect()
        torch.cuda.empty_cache()
        loaded = load_ocr("eager", quant)
        return loaded, "eager"


torch.cuda.reset_peak_memory_stats()
model = None
QUANT = None
ATTN = None
for candidate in (None, "8bit", "4bit"):
    try:
        model, ATTN = try_load(candidate)
        QUANT = candidate
        break
    except Exception as exc:
        if not is_oom(exc) and candidate is None:
            # Non-OOM on the first attempt (e.g. SDPA then eager both failed): stop.
            raise
        label = candidate or "float16"
        nxt = {None: "8-bit", "8bit": "4-bit"}.get(candidate)
        print(f"{label} load failed ({exc})" + (f"; trying {nxt}." if nxt else "."))
        model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("OCRia-1 did not fit in float16, 8-bit, or 4-bit on this GPU.")

processor = AutoProcessor.from_pretrained(MODEL_ID)
n_params = sum(p.numel() for p in model.parameters())
print("class:", type(model).__name__)
print(f"parameters: {n_params / 1e9:.2f}B | quant: {QUANT or 'float16'} | attention: {ATTN}")
print_vram("after load")


## 2. Helpers and sample pages

`transcribe(image)` follows the card: `prompt.txt` as system text, the RGB page as the only user content, `enable_thinking=False`, greedy `generate`. The card wraps the page in a markdown fence; the helper strips that fence.

Pages, all public URLs already used in this repo or on the model card:

* **Receipt** — SROIE ticket from [`hf-internal-testing/fixtures_ocr`](https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr) (same JPEG as LightOnOCR / MiniCPM-OCR / Unlimited-OCR / Xiaomi-OCR-0).
* **Table** — page 8 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (Table 2), the same PDF page as the LightOnOCR notebooks.
* **Chart** — the card's line-chart example (`assets/chart_line_page.jpg`): three series over five years, which OCRia-1 should write as an HTML table of the 15 data points.

Images are thumbnail-capped at **1600 px** on the long edge (~150 DPI for a letter page). The card's 4.1 MP budget is applied as a second cap.


In [ ]:
import io
import re
import time

import requests
from IPython.display import HTML, display
from PIL import Image

UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
RECEIPT_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
PAPER_URL = "https://arxiv.org/pdf/1706.03762"
CHART_URL = "https://huggingface.co/llamaindex/OCRia-1/resolve/main/assets/chart_line_page.jpg"
FENCE = re.compile(r"```markdown\n(.*?)(?:\n```|$)", re.S)
TABLE = re.compile(r"<table[\s\S]*?</table>", re.I)


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def prepare_page(image):
    image = image.convert("RGB")
    image.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
    pixels = image.width * image.height
    if pixels > MAX_PIXELS:
        scale = (MAX_PIXELS / pixels) ** 0.5
        image = image.resize(
            (max(1, round(image.width * scale)), max(1, round(image.height * scale))),
            Image.LANCZOS,
        )
    return image


def load_image_url(url):
    return prepare_page(Image.open(io.BytesIO(fetch_bytes(url))))


def render_pdf_page(url, page_index, dpi=150):
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(fetch_bytes(url))
    image = pdf[page_index].render(scale=dpi / 72).to_pil()
    return prepare_page(image)


def strip_fence(text):
    match = FENCE.search(text)
    return match.group(1) if match else text


def to_model(inputs):
    device = next((p.device for p in model.parameters() if p.device.type == "cuda"), torch.device("cuda"))
    inputs = inputs.to(device)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=DTYPE)
    return inputs


def transcribe(image, max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image", "image": image}]},
    ]
    # Card Transformers snippet: enable_thinking=False on apply_chat_template.
    # Same flag the vLLM extra_body passes as chat_template_kwargs.
    try:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        )
    except TypeError:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            chat_template_kwargs={"enable_thinking": False},
        )
    inputs = to_model(inputs)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output_ids[0, inputs["input_ids"].shape[1] :]
    text = processor.decode(generated, skip_special_tokens=True)
    print_vram("after generate")
    return strip_fence(text)


def show_tables(markdown):
    tables = TABLE.findall(markdown)
    if not tables:
        print("No <table> blocks in the output.")
        return
    for i, table in enumerate(tables, start=1):
        print(f"HTML table {i} of {len(tables)}")
        display(HTML(table))


receipt = load_image_url(RECEIPT_URL)
table_page = render_pdf_page(PAPER_URL, page_index=7)  # printed page 8, Table 2
chart_page = load_image_url(CHART_URL)
print("receipt", receipt.size, "| table page", table_page.size, "| chart", chart_page.size)


### Demo A — receipt

SROIE-style ticket (same JPEG as the other OCR notebooks). OCRia-1 should transcribe merchant, lines, and totals as Markdown. Display the page, then the model text.


In [ ]:
display(receipt)
receipt_text = transcribe(receipt, max_new_tokens=2048)
print(receipt_text)
show_tables(receipt_text)


### Demo B — results table (HTML)

Page 8 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (arXiv:1706.03762), Table 2 — the same page as [`LightOnOCR-3-4B-document-ocr.ipynb`](LightOnOCR-3-4B-document-ocr.ipynb). The card writes tables as HTML `<table>` (with `colspan` / `rowspan` when needed). The cell prints the Markdown and renders each HTML table with `IPython.display.HTML`.


In [ ]:
display(table_page)
table_text = transcribe(table_page, max_new_tokens=4096)
print(table_text)
show_tables(table_text)


### Demo C — chart → data table

The card's line-chart example (`assets/chart_line_page.jpg`). The system prompt requires every chart to become an HTML `<table>` of the plotted values, not a description of the figure. Render that table below the transcription.


In [ ]:
display(chart_page)
chart_text = transcribe(chart_page, max_new_tokens=4096)
print(chart_text)
show_tables(chart_text)


## 3. Caveats and cleanup

* **Transformers, not vLLM.** The card's throughput path is vLLM 0.30+ with thinking off. This notebook is the published `AutoModelForImageTextToText` snippet on a free T4.
* **dtype.** Saved weights are BF16. T4 has no native bfloat16. Default load is **float16**; 8-bit then 4-bit only if that OOMs. Do not set `torch.bfloat16` on this GPU.
* **Thinking off.** Pass `enable_thinking=False` (chat-template kwarg). The Qwen3.5 template defaults thinking on.
* **One page, image only.** No user text. Do not swap in another system prompt — the card says other prompts change the output format.
* **Resolution / tokens.** Card examples: 150 DPI greedy. Quality recipe: 300 DPI and `max_tokens` 24,576. This T4 path: max side **1600 px**, `max_new_tokens` **4096**.
* **Attention.** SDPA, or eager if SDPA rejects Qwen3.5 linear-attention layers. Not FlashAttention 2. Optional CUDA kernels (`flash-linear-attention`, `causal-conv1d`) are not installed; Transformers uses the PyTorch fallback.
* **GGUF.** [`llamaindex/OCRia-1-GGUF`](https://huggingface.co/llamaindex/OCRia-1-GGUF) is the llama.cpp / Ollama alternative. Not loaded here.
* **Licence.** OCRia-1 Model and Code License (`other`): free for research; free for commercial use by orgs under $2M revenue **and** $2M funding. Competing document-parsing products need a commercial license. See the [card](https://huggingface.co/llamaindex/OCRia-1#license).
* **Published scores are not rerun.** ParseBench / olmOCR-bench / PulseBench-Tab / fr-bench-pdf2md figures on the card are the authors' numbers.

**Related notebooks:** [LightOnOCR-3 4B](LightOnOCR-3-4B-document-ocr.ipynb), [LightOnOCR-3 0.8B](LightOnOCR-3-0.8B-document-ocr.ipynb), [Xiaomi-OCR-0](Xiaomi-OCR-0-document-ocr.ipynb).

Free the GPU before you open another notebook. The next cell prints `torch.cuda.max_memory_allocated` for this session.


In [ ]:
print_vram("before cleanup")
print(f"torch.cuda.max_memory_allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
try:
    del model, processor
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("OCRia-1 released. Re-run the load cell if you want another page.")
